In [ ]:
import pandas as pd
from google.cloud import bigquery
import os

# Client setup
bq_client = bigquery.Client()

print("Environment ready.")

In [ ]:
%%bigquery df
SELECT 
    p.category,
    p.brand,
    COUNT(oi.order_id) as total_orders,
    ROUND(SUM(oi.sale_price), 2) as total_revenue_usd,
    ROUND(AVG(oi.sale_price), 2) as avg_item_price
FROM `bigquery-public-data.thelook_ecommerce.order_items` oi
JOIN `bigquery-public-data.thelook_ecommerce.products` p 
    ON oi.product_id = p.id
WHERE oi.status = 'Complete' -- Only count completed sales
GROUP BY p.category, p.brand
ORDER BY total_revenue_usd DESC
LIMIT 2000

In [ ]:
# See the top 5 rows
df.head()

In [ ]:
# Group by Category to see which generates the most money
category_summary = df.groupby('category')[['total_revenue_usd', 'total_orders']].sum().sort_values(by='total_revenue_usd', ascending=False)

print("Top 5 Performing Categories:")
print(category_summary.head(5))

In [ ]:
# Quick bar chart of top categories
category_summary.head(10).plot(kind='bar', y='total_revenue_usd', title='Revenue by Category')

In [ ]:
BUCKET_NAME = "thelook-reports-2025-YourNameHere" # <--- CHANGE THIS NAME
REGION = "us-central1"

# Create the bucket using gsutil
!gsutil mb -l $REGION gs://$BUCKET_NAME

In [ ]:
# Define the filename
file_path = f"gs://{BUCKET_NAME}/monthly_category_report.csv"

# Write DataFrame directly to GCS
category_summary.to_csv(file_path)

print(f"Report successfully uploaded to: {file_path}")

In [ ]:
!gsutil ls -l gs://$BUCKET_NAME

In [ ]:
import warnings
import vertexai
from vertexai.generative_models import GenerativeModel
import pandas as pd
from IPython.display import display, Markdown

# 1. Clean up the output (Hide deprecation warnings)
warnings.filterwarnings('ignore')

# 2. Initialize Vertex AI
# We use 'us-central1' as it is the cheapest/standard region
vertexai.init(location="us-central1")

# 3. Load the Model
# 'flash' is faster and uses fewer tokens than 'pro'
model = GenerativeModel("gemini-2.5-pro")

print("✅ Model loaded successfully.")

In [ ]:
# 1. Load Data from GCS
# (Assumes BUCKET_NAME was defined in previous steps)
report_url = f"gs://{BUCKET_NAME}/monthly_category_report.csv"
df = pd.read_csv(report_url)

# 2. Optimize Context (Token Savings)
# We convert only the top 30 rows to Markdown.
# This gives the AI enough context without reading the whole file.
data_context = df.head(30).to_markdown(index=False)

# 3. Start the Chat Session
chat = model.start_chat()

# 4. Prime the Model (System Prompt)
# We give strict instructions for formatting here.
system_instruction = f"""
You are a Senior Data Analyst. Analyze the table below.

DATA:
{data_context}

RULES:
1. Answer strictly based on the provided data.
2. Use bullet points (*) for lists.
3. Keep answers concise (under 50 words if possible).
4. Format key metrics in **bold**.
"""

# Send this silently to set the stage
chat.send_message(system_instruction)

print("✅ Data ingested. The AI is ready for questions!")

In [ ]:
print("--- 💬 Chat with your Data (Type 'exit' to quit) ---")

while True:
    # 1. Get User Input
    user_query = input("You: ")
    
    # 2. Check for Exit
    if user_query.lower() in ['exit', 'quit', 'done']:
        print("\n🛑 Session ended.")
        break
    
    # 3. Show "Thinking" Placeholder
    # 'end="\r"' keeps the cursor on the same line so we can overwrite it later
    print("⏳ AI is analyzing the data...", end="\r")
    
    try:
        # 4. Get Response
        response = chat.send_message(user_query)
        
        # 5. Clear the "Thinking" text
        print(" " * 30, end="\r") 
        
        # 6. Display Clean Output
        # We use 'Markdown' to render the bolding and lists nicely
        display(Markdown(f"**Gemini:**\n{response.text}"))
        print("-" * 50)
        
    except Exception as e:
        print(f"Error: {e}")

<!-- Sample Questions:
What are the top 5 categories by total revenue?
How many total orders did the 'Jeans' category get?
Is there a relationship between high average item price and total revenue? Do expensive items sell better? -->


<!-- Sample Questions:
What are the top 5 categories by total revenue?
How many total orders did the 'Jeans' category get?
Is there a relationship between high average item price and total revenue? Do expensive items sell better? -->
